In [1]:
import gymnasium as gym
import numpy as np
import pickle
import os
from tqdm.notebook import tqdm
import gym_snakegame
from rewards import SnakeRewardWrapper

# Experimental Extension: Curriculum Learning Strategy

The curriculum is staged as follows:
1. **Stage 1:** 15,000 episodes on a $6 \times 6$ board.
2. **Stage 2:** 15,000 episodes on a $8 \times 8$ board.
3. **Stage 3:** 20,000 episodes on the final $10 \times 10$ board with a refined $\epsilon$-floor. \

This strategy is superior for this reinforcement learning task for one main reason:
* On smaller grids ($6 \times 6$), the probability of the agent randomly encountering a food item is significantly higher. This allows the agent to populate its Q-table with a policy for pathfinding and collision avoidance before the state space expands on larger boards.


In [4]:


def get_allocentric_state(env):
    snake = env.unwrapped.snake
    if len(snake) == 0: return (0,)*11
    
    board = env.unwrapped.board
    head = snake[-1]
    target_val = env.unwrapped.ITEM
    bs = env.unwrapped.board_size
    target_pos = np.argwhere(board == target_val)[0]

    def is_unsafe(pos):
        r, c = pos
        if not (0 <= r < bs and 0 <= c < bs): return True
        return 0 < board[r, c] < target_val

    return (
        is_unsafe((head[0]-1, head[1])), is_unsafe((head[0]+1, head[1])),
        is_unsafe((head[0], head[1]-1)), is_unsafe((head[0], head[1]+1)),
        target_pos[0] < head[0], target_pos[0] > head[0],
        target_pos[1] < head[1], target_pos[1] > head[1],
        env.unwrapped.prev_action == 0, env.unwrapped.prev_action == 1,
        env.unwrapped.prev_action == 2
    )

def get_egocentric_state(env):
    snake = env.unwrapped.snake
    if len(snake) == 0: return (0,)*6
    board = env.unwrapped.board
    head = snake[-1]
    target_val = env.unwrapped.ITEM
    bs = env.unwrapped.board_size
    target_pos = np.argwhere(board == target_val)[0]
    prev_action = env.unwrapped.prev_action 

    dirs = {0: (-1, 0), 1: (0, 1), 2: (1, 0), 3: (0, -1)}
    curr_dir = dirs.get(prev_action, (0, 1)) 

    forward = curr_dir
    left = (-curr_dir[1], curr_dir[0])
    right = (curr_dir[1], -curr_dir[0])

    def is_unsafe(offset):
        r, c = head[0] + offset[0], head[1] + offset[1]
        if not (0 <= r < bs and 0 <= c < bs): return True
        return 0 < board[r, c] < target_val

    food_vec = (target_pos[0] - head[0], target_pos[1] - head[1])
    food_forward = np.sign(food_vec[0]*forward[0] + food_vec[1]*forward[1])
    food_lateral = np.sign(food_vec[0]*right[0] + food_vec[1]*right[1])

    return (is_unsafe(forward), is_unsafe(left), is_unsafe(right), 
            food_forward, food_lateral, prev_action)

In [6]:
class QLearningAgent:
    def __init__(self, alpha=0.1, gamma=0.95):
        self.q_table = {}
        self.alpha, self.gamma = alpha, gamma

    def get_action(self, state, epsilon):
        if np.random.rand() < epsilon:
            return np.random.randint(4)
        q_values = [self.q_table.get((state, i), 0.0) for i in range(4)]
        return np.argmax(q_values)

    def update(self, s, a, r, s_n):
        q_max = max([self.q_table.get((s_n, i), 0.0) for i in range(4)])
        curr = self.q_table.get((s, a), 0.0)
        self.q_table[(s, a)] = curr + self.alpha * (r + self.gamma * q_max - curr)

In [7]:
stages = [{"size": 6, "ep": 10000}, {"size": 8, "ep": 15000}, {"size": 10, "ep": 25000}]
rewards = ["sparse", "dense", "survivalist"]
configs = {"Allocentric": get_allocentric_state, "Egocentric": get_egocentric_state}
os.makedirs("final_results/models", exist_ok=True)

for r_type in rewards:
    for name, state_func in configs.items():
        agent = QLearningAgent()
        epsilon, decay = 1.0, 0.9999
        
        for stage in stages:
            env = SnakeRewardWrapper(gym.make("gym_snakegame/SnakeGame-v0", board_size=stage["size"]), r_type)
            for _ in tqdm(range(stage["ep"]), desc=f"{r_type} | {name} | {stage['size']}x{stage['size']}"):
                obs, _ = env.reset(); s = str(state_func(env)); done = False
                while not done:
                    action = agent.get_action(s, epsilon)
                    _, r, term, trunc, _ = env.step(action); sn = str(state_func(env))
                    agent.update(s, action, r, sn); s = sn; done = term or trunc
                    epsilon = max(0.01, epsilon * decay)
        
        with open(f"final_results/models/{name}_{r_type}_Curriculum_q_table.pkl", "wb") as f:
            pickle.dump(agent.q_table, f)

sparse | Allocentric | 6x6:   0%|          | 0/10000 [00:00<?, ?it/s]

sparse | Allocentric | 8x8:   0%|          | 0/15000 [00:00<?, ?it/s]

sparse | Allocentric | 10x10:   0%|          | 0/25000 [00:00<?, ?it/s]

sparse | Egocentric | 6x6:   0%|          | 0/10000 [00:00<?, ?it/s]

sparse | Egocentric | 8x8:   0%|          | 0/15000 [00:00<?, ?it/s]

...

survivalist | Allocentric | 8x8:   0%|          | 0/15000 [00:00<?, ?it/s]

survivalist | Allocentric | 10x10:   0%|          | 0/25000 [00:00<?, ?it/s]

survivalist | Egocentric | 6x6:   0%|          | 0/10000 [00:00<?, ?it/s]

survivalist | Egocentric | 8x8:   0%|          | 0/15000 [00:00<?, ?it/s]

survivalist | Egocentric | 10x10:   0%|          | 0/25000 [00:00<?, ?it/s]